# 07b — Test evaluation: Siamese concat (class-weighted CE, 40 epochs)

In [ ]:
BRANCH = "main"

%cd /tmp
!rm -rf damage-detection
!git clone -b {BRANCH} https://github.com/azrairemd/damage-detection.git
%cd damage-detection
!pip install -q -r requirements.txt
!pip install -q -e .

/tmp
Cloning into 'damage-detection'...
remote: Enumerating objects: 395, done.
remote: Counting objects: 100% (234/234), done.
remote: Compressing objects: 100% (192/192), done.
remote: Total 395 (delta 94), reused 137 (delta 41), pack-reused 161 (from 1)
Receiving objects: 100% (395/395), 17.41 MiB | 31.38 MiB/s, done.
Resolving deltas: 100% (173/173), done.
/tmp/damage-detection
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for damage-detection (pyproject.toml) ... done


In [2]:
import json

import torch

from src.data.dataset import XBDDataset
from src.evaluate import evaluate, load_model
from src.models.siamese_unet import SiameseUNet
from src.utils.config import load_config

In [3]:
CE_PATH = "/kaggle/input/notebooks/azra41221/06c-train-siamese-fd40/outputs/siamese_ce_long/best.pt"

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [5]:
ce_model = load_model(SiameseUNet(encoder="resnet34", fusion="concat"), CE_PATH, device)

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

## 1. Sanity check on the validation set

In [6]:
config = load_config("configs/siamese_ce_long.yaml")
data = config["data"]

val_ds = XBDDataset(data["val_split"], data["xbd_root"], data["mask_root"], train=False)
val_overall, _, _ = evaluate(ce_model, val_ds, device)

print("Validation score:", round(val_overall["score"], 4), " (during training: 0.7401)")
print("Localization F1: ", round(val_overall["f1_loc"], 4), " (during training: 0.7927)")
print("Damage F1:       ", round(val_overall["f1_dmg"], 4), " (during training: 0.7176)")

# stop here if this is not the 40-epoch CE checkpoint
assert abs(val_overall["score"] - 0.7401) < 0.001, "Wrong checkpoint: do NOT evaluate on the test set"

Validation score: 0.7401  (during training: 0.7401)
Localization F1:  0.7927  (during training: 0.7927)
Damage F1:        0.7176  (during training: 0.7176)


## 2. Test set evaluation (run once)

In [ ]:
test_ds = XBDDataset("splits/test.csv", data["xbd_root"], data["mask_root"], train=False)
print("Test images:", len(test_ds))

overall, per_type, confusion = evaluate(ce_model, test_ds, device)
results = {"siamese_ce": {"overall": overall, "per_type": per_type, "confusion": confusion.tolist()}}
print(f"siamese_ce test score {overall['score']:.4f}  loc {overall['f1_loc']:.4f}  dmg {overall['f1_dmg']:.4f}")

with open("/kaggle/working/test_results_ce.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved to /kaggle/working/test_results_ce.json")